In [0]:
#  category_topic_dynamic_rules_notebook.py

In [0]:
# Databricks notebook source
# COMMAND ----------
# 1) Config

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
ENDPOINT = "databricks-gpt-5-4"

PROMPT_VERSION = "category_topic_dynamic_rules_v1"
SAVE_DB = "sandbox.z_jungryo_lee"

TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-06. 리모컨 사용성"
TARGET_SC_MEASUREMENT = 1
RULE_SC_MEASUREMENTS = [1, -1]

RULE_PROFILE_TABLE = f"{SAVE_DB}.category_topic_rule_profile_{PROMPT_VERSION}"
TOPIC_POOL_TABLE = f"{SAVE_DB}.category_topic_pool_{PROMPT_VERSION}"
DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_{PROMPT_VERSION}"
SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_{PROMPT_VERSION}"

MAX_SAMPLE_ROWS = 1000
MAX_FINAL_TOPICS = 17
MIN_ROWS_PER_TOPIC = 10
CLASSIFY_BATCH_SIZE = 25
MAX_TOKENS = 2200
MAX_RETRIES = 3
BACKOFF_BASE = 1.8
RATE_LIMIT_SECONDS = 0.4

print(
    f"[CONFIG] endpoint={ENDPOINT}, cate_1={TARGET_CATE_1_DEPTH}, cate_2={TARGET_CATE_2_DEPTH}, "
    f"target_sc={TARGET_SC_MEASUREMENT}, max_topics={MAX_FINAL_TOPICS}"
)

In [0]:
# COMMAND ----------
# 2) Helpers

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)

    for candidate in [text]:
        try:
            return json.loads(candidate)
        except Exception:
            pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            candidate = re.sub(r",\s*}", "}", candidate)
            candidate = re.sub(r",\s*]", "]", candidate)
            return json.loads(candidate)

    raise ValueError(f"Invalid JSON: {text[:1000]}")


def chunk_list(items: List[Any], size: int) -> List[List[Any]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def sc_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "긍정"
    if sc_measurement == -1:
        return "부정"
    return "기타"


def overall_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "전반적 긍정"
    if sc_measurement == -1:
        return "전반적 부정"
    return "전반적 평가"


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": max_tokens,
    }

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            print(f"[LLM CALL] attempt={attempt + 1}/{MAX_RETRIES}")
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])
            if isinstance(resp, str):
                return extract_json(resp)
            raise ValueError(f"Unexpected response schema: {resp}")
        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            if resp is not None:
                print("[LLM RAW RESPONSE PREVIEW]")
                print(str(resp)[:2000])
            wait_sec = BACKOFF_BASE ** attempt
            print(f"[LLM RETRY WAIT] {wait_sec:.2f}s")
            time.sleep(wait_sec)

    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


def save_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] overwrite -> {table_name}")
    else:
        print(f"[SAVE] create -> {table_name}")
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def has_specific_reason(text: str, clue_keywords: List[str]) -> bool:
    memo = clean_text(text).lower()
    return any(clean_text(k).lower() in memo for k in clue_keywords if clean_text(k))


def should_be_overall(text: str, clue_keywords: List[str], generic_terms: List[str]) -> bool:
    memo = clean_text(text).lower()
    if not memo:
        return False
    if len(memo) > 20:
        return False
    if has_specific_reason(memo, clue_keywords):
        return False
    return any(clean_text(term).lower() in memo for term in generic_terms if clean_text(term))

In [0]:
# COMMAND ----------
# 3) Load Sample Memos

def load_sample_df(sc_measurement: int, max_rows: int = MAX_SAMPLE_ROWS):
    query = f"""
    with base as (
        select distinct memo
        from {SOURCE_TABLE}
        where 1=1
          and cate_1_depth = '{TARGET_CATE_1_DEPTH}'
          and cate_2_depth = '{TARGET_CATE_2_DEPTH}'
          and sc_measurement = {sc_measurement}
          and memo is not null
          and length(trim(memo)) > 0
    ),
    sampled as (
        select
            memo,
            row_number() over (
                order by md5(
                    concat(
                        coalesce(memo, ''),
                        '||',
                        '{TARGET_CATE_1_DEPTH}',
                        '||',
                        '{TARGET_CATE_2_DEPTH}',
                        '||',
                        cast({sc_measurement} as string),
                        '||seed_20260420'
                    )
                )
            ) as rn
        from base
    )
    select memo
    from sampled
    where rn <= {max_rows}
    order by rn
    """
    return spark.sql(query).withColumn("_row_id", F.monotonically_increasing_id()).cache()


sample_df = load_sample_df(TARGET_SC_MEASUREMENT, MAX_SAMPLE_ROWS)
print(f"[LOAD] target sample rows = {sample_df.count()}")
display(sample_df.limit(20))


In [0]:
# 4) Generate Dynamic Rule Profiles By sc_measurement

def build_rule_profile_messages(sc_measurement: int, sample_memos: List[str]) -> List[Dict[str, str]]:
    label = sc_label(sc_measurement)
    overall = overall_label(sc_measurement)
    system = f"""
You are a VOC rule designer for TV review topic classification.

Your task:
- Read review memos from one fixed category and one fixed polarity.
- Build reusable rule components for topic classification.

Category:
- cate_1_depth = {TARGET_CATE_1_DEPTH}
- cate_2_depth = {TARGET_CATE_2_DEPTH}
- polarity = {label}

Design outputs:
- overall_topic_label
- clue_keywords: category-specific clue words or short phrases that indicate a specific reason
- generic_terms: very generic sentiment terms that can belong to {overall}
- overall_usage_rule
- specific_reason_rule
- non_overall_examples: examples of memo patterns that must NOT belong to {overall}

Rules:
- clue_keywords must be specialized for this category, not generic for all TV reviews.
- generic_terms must be suitable for this polarity only.
- {overall} should be extremely narrow.
- {overall} must be limited to ultra-short sentiment-only memos where no usable reason can be inferred.
- If a memo mentions usability, design, control, feature, convenience, responsiveness, app use, buttons,
  smart behavior, or any functional impression, it must NOT be treated as {overall}.
- If a memo contains any usable reason, feature, benefit, complaint, or context clue, it must not be treated as {overall}.
- Keep clue_keywords compact and reusable.
- Keep non_overall_examples short.

Return only JSON:
{{
  "overall_topic_label": "",
  "clue_keywords": [""],
  "generic_terms": [""],
  "overall_usage_rule": "",
  "specific_reason_rule": "",
  "non_overall_examples": [""]
}}
"""
    user = "Review memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


rule_rows: List[Dict[str, Any]] = []
for sc in RULE_SC_MEASUREMENTS:
    rule_sample_df = load_sample_df(sc)
    rule_sample_memos = [r["memo"] for r in rule_sample_df.select("memo").toLocalIterator()]
    rule_result = call_llm(build_rule_profile_messages(sc, rule_sample_memos))
    rule_rows.append(
        {
            "cate_1_depth": TARGET_CATE_1_DEPTH,
            "cate_2_depth": TARGET_CATE_2_DEPTH,
            "sc_measurement": sc,
            "overall_topic_label": clean_text(rule_result.get("overall_topic_label")) or overall_label(sc),
            "clue_keywords_json": json.dumps(rule_result.get("clue_keywords", [])[:60], ensure_ascii=False),
            "generic_terms_json": json.dumps(rule_result.get("generic_terms", [])[:30], ensure_ascii=False),
            "overall_usage_rule": clean_text(rule_result.get("overall_usage_rule")),
            "specific_reason_rule": clean_text(rule_result.get("specific_reason_rule")),
            "non_overall_examples_json": json.dumps(rule_result.get("non_overall_examples", [])[:10], ensure_ascii=False),
        }
    )
    rule_sample_df.unpersist()
    time.sleep(RATE_LIMIT_SECONDS)

rule_profile_df = spark.createDataFrame(pd.DataFrame(rule_rows))
save_table(rule_profile_df, RULE_PROFILE_TABLE)
display(rule_profile_df)


In [0]:
# 5) Load Dynamic Rule Profile For Target sc_measurement

rule_profile_row = (
    rule_profile_df
    .where(F.col("sc_measurement") == F.lit(TARGET_SC_MEASUREMENT))
    .limit(1)
    .collect()[0]
)

OVERALL_TOPIC_LABEL = rule_profile_row["overall_topic_label"]
CLUE_KEYWORDS = json.loads(rule_profile_row["clue_keywords_json"]) if rule_profile_row["clue_keywords_json"] else []
GENERIC_TERMS = json.loads(rule_profile_row["generic_terms_json"]) if rule_profile_row["generic_terms_json"] else []
OVERALL_USAGE_RULE = clean_text(rule_profile_row["overall_usage_rule"])
SPECIFIC_REASON_RULE = clean_text(rule_profile_row["specific_reason_rule"])
NON_OVERALL_EXAMPLES = json.loads(rule_profile_row["non_overall_examples_json"]) if rule_profile_row["non_overall_examples_json"] else []

print("[RULE PROFILE]")
print("overall_topic_label =", OVERALL_TOPIC_LABEL)
print("clue_keywords =", CLUE_KEYWORDS[:20])
print("generic_terms =", GENERIC_TERMS)
print("overall_usage_rule =", OVERALL_USAGE_RULE)
print("specific_reason_rule =", SPECIFIC_REASON_RULE)
print("non_overall_examples =", NON_OVERALL_EXAMPLES)


In [0]:
# 6) Build Topic Pool Using Dynamic Rule Profile

sample_memos = [r["memo"] for r in sample_df.select("memo").toLocalIterator()]

topic_pool_system = f"""
You are a VOC taxonomy designer for TV review topic classification.

Your task:
- Read up to 1000 review memos from one fixed category and polarity.
- Build ONE fixed topic pool that explains WHY users evaluated the category as {sc_label(TARGET_SC_MEASUREMENT)}.

Category:
- cate_1_depth = {TARGET_CATE_1_DEPTH}
- cate_2_depth = {TARGET_CATE_2_DEPTH}

Hard rules:
- Final topic count must be at least 7 and at most {MAX_FINAL_TOPICS}.
- One mandatory topic must be '{OVERALL_TOPIC_LABEL}'.
- {OVERALL_USAGE_RULE}
- {SPECIFIC_REASON_RULE}
- '{OVERALL_TOPIC_LABEL}' must be extremely narrow and only used when no reason can be inferred at all.
- If a memo contains any feature, usability, design, control, convenience, function, response, or app-related clue,
  it must NOT belong to '{OVERALL_TOPIC_LABEL}'.
- Category-specific clue keywords that indicate specific reasons:
  {json.dumps(CLUE_KEYWORDS, ensure_ascii=False)}
- Examples that are NOT '{OVERALL_TOPIC_LABEL}':
  {json.dumps(NON_OVERALL_EXAMPLES, ensure_ascii=False)}
- All non-overall topics should describe a specific reason.
- Topic labels must be Korean.
- topic must be concise noun-like or service/function-like wording.
- description must be a short Korean explanation of what kind of memo belongs there.
- Avoid duplicates and near-synonyms.
- Leave room for one catch-all '기타' topic during post-processing; do not over-fragment topics.

Return only JSON:
{{
  "topics": [
    {{
      "topic": "",
      "description": "",
      "representative_memos": [""]
    }}
  ]
}}
"""

topic_pool_user = "Sample memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])

topic_pool_result = call_llm(
    [
        {"role": "system", "content": clean_text(topic_pool_system)},
        {"role": "user", "content": clean_text(topic_pool_user)},
    ]
)

topic_rows: List[Dict[str, Any]] = []
for order_no, topic in enumerate(topic_pool_result.get("topics", [])[:MAX_FINAL_TOPICS], start=1):
    if not isinstance(topic, dict):
        continue
    topic_rows.append(
        {
            "topic_order": order_no,
            "topic": clean_text(topic.get("topic")),
            "description": clean_text(topic.get("description")),
            "representative_memos_json": json.dumps(topic.get("representative_memos", [])[:5], ensure_ascii=False),
        }
    )

topic_pool_df = spark.createDataFrame(pd.DataFrame(topic_rows))
save_table(topic_pool_df, TOPIC_POOL_TABLE)
display(topic_pool_df)

In [0]:
# 7) Classify Using Dynamic Rule Profile

topic_pool_payload = [
    {
        "topic": r["topic"],
        "description": r["description"],
    }
    for r in topic_pool_df.orderBy("topic_order").toLocalIterator()
]


def build_classify_messages(batch_rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    system = f"""
You are a VOC classifier for TV review topic classification.

Classify each memo into exactly one topic from the fixed topic list.
Every memo must belong to exactly one topic.

Rules:
- The task is to identify WHY the writer evaluated {TARGET_CATE_2_DEPTH} as {sc_label(TARGET_SC_MEASUREMENT)}.
- Overall topic is '{OVERALL_TOPIC_LABEL}'.
- {OVERALL_USAGE_RULE}
- {SPECIFIC_REASON_RULE}
- '{OVERALL_TOPIC_LABEL}' is allowed only for ultra-short sentiment-only memos with no usable reason.
- If a memo contains any specific reason, function, usability signal, design signal, or feature impression,
  it must be assigned to a non-overall topic.
- clue keywords indicating specific reasons:
  {json.dumps(CLUE_KEYWORDS, ensure_ascii=False)}
- examples that are NOT '{OVERALL_TOPIC_LABEL}':
  {json.dumps(NON_OVERALL_EXAMPLES, ensure_ascii=False)}
- Do not invent any new topic.
- For multilingual memos, understand them and reason in Korean.
- explanation must be a short Korean sentence explaining why the memo belongs to the topic.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Fixed topics:\n"
        + json.dumps(topic_pool_payload, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


def build_reclassify_without_overall_message(row: Dict[str, Any]) -> List[Dict[str, str]]:
    specific_topic_payload = [topic for topic in topic_pool_payload if clean_text(topic["topic"]) != OVERALL_TOPIC_LABEL]
    system = f"""
You are a VOC classifier for TV review topic classification.

This memo was incorrectly over-generalized as '{OVERALL_TOPIC_LABEL}'.
Reclassify it using only the non-general topics below.

Rules:
- Choose exactly one non-general topic.
- The memo contains a specific reason and must not remain as '{OVERALL_TOPIC_LABEL}'.
- Use the clue keywords below as hints for specific reasons:
  {json.dumps(CLUE_KEYWORDS, ensure_ascii=False)}
- explanation must be a short Korean sentence explaining the specific reason.

Return only JSON:
{{
  "topic": "",
  "explanation": ""
}}
"""
    user = (
        "Allowed non-general topics:\n"
        + json.dumps(specific_topic_payload, ensure_ascii=False)
        + "\n\nMemo:\n"
        + json.dumps({"memo": clean_text(row["memo"])}, ensure_ascii=False)
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


classified_rows: List[Dict[str, Any]] = []
source_rows = [r.asDict(recursive=True) for r in sample_df.toLocalIterator()]

for batch in chunk_list(source_rows, CLASSIFY_BATCH_SIZE):
    batch_result = call_llm(build_classify_messages(batch))
    result_map = {
        str(item.get("row_id")): item
        for item in batch_result.get("results", [])
        if isinstance(item, dict)
    }

    for row in batch:
        mapped = result_map.get(str(row["_row_id"]), {})
        topic_raw = clean_text(mapped.get("topic"))
        explanation_raw = clean_text(mapped.get("explanation"))

        if (
            topic_raw == OVERALL_TOPIC_LABEL
            and has_specific_reason(row["memo"], CLUE_KEYWORDS)
            and not should_be_overall(row["memo"], CLUE_KEYWORDS, GENERIC_TERMS)
        ):
            retry_result = call_llm(build_reclassify_without_overall_message(row))
            retry_topic = clean_text(retry_result.get("topic"))
            retry_explanation = clean_text(retry_result.get("explanation"))
            if retry_topic:
                topic_raw = retry_topic
            if retry_explanation:
                explanation_raw = retry_explanation

        classified_rows.append(
            {
                "_row_id": row["_row_id"],
                "memo": row["memo"],
                "topic_raw": topic_raw,
                "explanation_raw": explanation_raw,
            }
        )
    time.sleep(RATE_LIMIT_SECONDS)

classified_df = spark.createDataFrame(pd.DataFrame(classified_rows))
display(classified_df.limit(20))


In [0]:
# 8) Merge Small Topics (<10) Into '기타'

topic_stats_df = classified_df.groupBy("topic_raw").agg(F.count("*").alias("topic_cnt"))
rare_topics = [r["topic_raw"] for r in topic_stats_df.where(F.col("topic_cnt") < F.lit(MIN_ROWS_PER_TOPIC)).collect()]
rare_topics_sorted = sorted([t for t in rare_topics if clean_text(t)])
rare_topic_label = f"기타({', '.join(rare_topics_sorted[:3])})" if rare_topics_sorted else "기타"

topic_desc_map = {
    r["topic"]: r["description"]
    for r in topic_pool_df.select("topic", "description").toLocalIterator()
}

final_rows: List[Dict[str, Any]] = []
for row in classified_df.toLocalIterator():
    row_dict = row.asDict(recursive=True)
    raw_topic = clean_text(row_dict["topic_raw"])
    raw_expl = clean_text(row_dict["explanation_raw"])

    if raw_topic in rare_topics_sorted:
        final_topic = rare_topic_label
        final_explanation = f"희소 주제 묶음: {raw_topic}" if raw_topic else "희소 주제 묶음"
    else:
        final_topic = raw_topic if raw_topic else "기타"
        final_explanation = raw_expl or topic_desc_map.get(final_topic, "")

    final_rows.append(
        {
            "_row_id": row_dict["_row_id"],
            "memo": row_dict["memo"],
            "topic": final_topic,
            "description": final_explanation,
        }
    )

detail_df = spark.createDataFrame(pd.DataFrame(final_rows))
display(detail_df.limit(20))


In [0]:
# COMMAND ----------
# 9) Load Full Population And Exclude Already-Classified Sample Rows

full_query = f"""
with base as (
    select distinct
        memo
    from {SOURCE_TABLE}
    where 1=1
      and cate_1_depth = '{TARGET_CATE_1_DEPTH}'
      and cate_2_depth = '{TARGET_CATE_2_DEPTH}'
      and sc_measurement = {TARGET_SC_MEASUREMENT}
      and memo is not null
      and length(trim(memo)) > 0
)
select memo
from base
"""

full_df = (
    spark.sql(full_query)
    .withColumn("_row_id", F.monotonically_increasing_id())
    .withColumn("row_key", F.md5(F.concat_ws("||", F.lit(TARGET_CATE_1_DEPTH), F.lit(TARGET_CATE_2_DEPTH), F.lit(str(TARGET_SC_MEASUREMENT)), F.col("memo"))))
    .cache()
)

sample_key_df = (
    sample_df
    .withColumn("row_key", F.md5(F.concat_ws("||", F.lit(TARGET_CATE_1_DEPTH), F.lit(TARGET_CATE_2_DEPTH), F.lit(str(TARGET_SC_MEASUREMENT)), F.col("memo"))))
    .select("row_key")
    .distinct()
)

remaining_df = (
    full_df.join(sample_key_df, on="row_key", how="left_anti")
    .drop("row_key")
    .cache()
)

print(f"[FULL] total rows = {full_df.count()}")
print(f"[SAMPLE] already classified sample rows = {sample_key_df.count()}")
print(f"[REMAINING] rows to classify = {remaining_df.count()}")

display(remaining_df.limit(20))


In [0]:
# COMMAND ----------
# 10) Resume / Checkpoint Config

FULL_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_full_{PROMPT_VERSION}"
FULL_SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_full_{PROMPT_VERSION}"

INTERMEDIATE_REMAINING_TABLE = f"{SAVE_DB}.category_topic_remaining_checkpoint_{PROMPT_VERSION}"
INTERMEDIATE_FAILED_TABLE = f"{SAVE_DB}.category_topic_remaining_failed_{PROMPT_VERSION}"

REMAINING_CLASSIFY_BATCH_SIZE = 40
MAX_ROW_RETRIES = 2

print(
    f"[CHECKPOINT CONFIG] batch_size={REMAINING_CLASSIFY_BATCH_SIZE}, "
    f"checkpoint_table={INTERMEDIATE_REMAINING_TABLE}, failed_table={INTERMEDIATE_FAILED_TABLE}"
)


In [0]:
# COMMAND ----------
# 11) Append Helper + Full Population Load

def append_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] append -> {table_name}")
        df.write.mode("append").format("delta").saveAsTable(table_name)
    else:
        print(f"[SAVE] create -> {table_name}")
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def empty_checkpoint_df():
    return spark.createDataFrame(
        [],
        "_row_id long, row_key string, memo string, topic_raw string, explanation_raw string, "
        "similarity_score double, classification_route string"
    )


def empty_failed_df():
    return spark.createDataFrame(
        [],
        "_row_id long, row_key string, memo string, error_stage string, error_message string"
    )


full_query = f"""
with base as (
    select distinct
        memo
    from {SOURCE_TABLE}
    where 1=1
      and cate_1_depth = '{TARGET_CATE_1_DEPTH}'
      and cate_2_depth = '{TARGET_CATE_2_DEPTH}'
      and sc_measurement = {TARGET_SC_MEASUREMENT}
      and memo is not null
      and length(trim(memo)) > 0
)
select memo
from base
"""

full_df = (
    spark.sql(full_query)
    .withColumn("_row_id", F.monotonically_increasing_id())
    .withColumn(
        "row_key",
        F.md5(
            F.concat_ws(
                "||",
                F.lit(TARGET_CATE_1_DEPTH),
                F.lit(TARGET_CATE_2_DEPTH),
                F.lit(str(TARGET_SC_MEASUREMENT)),
                F.col("memo"),
            )
        )
    )
    .cache()
)

sample_key_df = (
    sample_df
    .withColumn(
        "row_key",
        F.md5(
            F.concat_ws(
                "||",
                F.lit(TARGET_CATE_1_DEPTH),
                F.lit(TARGET_CATE_2_DEPTH),
                F.lit(str(TARGET_SC_MEASUREMENT)),
                F.col("memo"),
            )
        )
    )
    .select("row_key")
    .distinct()
)

if spark.catalog.tableExists(INTERMEDIATE_REMAINING_TABLE):
    processed_key_df = spark.table(INTERMEDIATE_REMAINING_TABLE).select("row_key").distinct()
    print(f"[RESUME] processed rows in checkpoint = {processed_key_df.count()}")
else:
    processed_key_df = empty_checkpoint_df().select("row_key")
    print("[RESUME] no checkpoint table found")

remaining_df = (
    full_df
    .join(sample_key_df, on="row_key", how="left_anti")
    .join(processed_key_df, on="row_key", how="left_anti")
    .cache()
)

print(f"[FULL] total distinct memos = {full_df.count()}")
print(f"[SAMPLE] sample memos = {sample_key_df.count()}")
print(f"[REMAINING] rows to classify now = {remaining_df.count()}")

display(remaining_df.limit(20))


In [0]:
# COMMAND ----------
# 12) Safe Single-Row Classifier

def classify_one_row_safely(row: Dict[str, Any]) -> Dict[str, Any]:
    last_error = None

    for _ in range(MAX_ROW_RETRIES):
        try:
            batch_result = call_llm(build_classify_messages([row]))
            results = [item for item in batch_result.get("results", []) if isinstance(item, dict)]
            mapped = results[0] if results else {}

            topic_raw = clean_text(mapped.get("topic"))
            explanation_raw = clean_text(mapped.get("explanation"))

            if (
                topic_raw == OVERALL_TOPIC_LABEL
                and has_specific_reason(row["memo"], CLUE_KEYWORDS)
                and not should_be_overall(row["memo"], CLUE_KEYWORDS, GENERIC_TERMS)
            ):
                retry_result = call_llm(build_reclassify_without_overall_message(row))
                retry_topic = clean_text(retry_result.get("topic"))
                retry_explanation = clean_text(retry_result.get("explanation"))
                if retry_topic:
                    topic_raw = retry_topic
                if retry_explanation:
                    explanation_raw = retry_explanation

            return {
                "_row_id": row["_row_id"],
                "row_key": row["row_key"],
                "memo": row["memo"],
                "topic_raw": topic_raw,
                "explanation_raw": explanation_raw,
                "similarity_score": None,
                "classification_route": "llm_row_retry",
            }
        except Exception as exc:
            last_error = exc
            time.sleep(RATE_LIMIT_SECONDS)

    raise RuntimeError(f"single_row_failed: {repr(last_error)}")


In [0]:
# COMMAND ----------
# 13) Resume-Safe Remaining Classification (with progress logs)

remaining_rows = [r.asDict(recursive=True) for r in remaining_df.toLocalIterator()]

total_rows_to_process = len(remaining_rows)
total_batches = (total_rows_to_process + REMAINING_CLASSIFY_BATCH_SIZE - 1) // REMAINING_CLASSIFY_BATCH_SIZE

print(f"[START] rows_to_process={total_rows_to_process}, total_batches={total_batches}")

saved_total = 0
failed_total = 0
processed_total = 0

for batch_no, batch in enumerate(chunk_list(remaining_rows, REMAINING_CLASSIFY_BATCH_SIZE), start=1):
    batch_saved_rows = []
    batch_failed_rows = []

    batch_size = len(batch)
    batch_start_time = time.time()

    print(
        f"[BATCH START] batch_no={batch_no}/{total_batches}, "
        f"batch_size={batch_size}, processed_so_far={processed_total}"
    )

    try:
        batch_result = call_llm(build_classify_messages(batch))
        result_map = {
            str(item.get("row_id")): item
            for item in batch_result.get("results", [])
            if isinstance(item, dict)
        }

        for row in batch:
            mapped = result_map.get(str(row["_row_id"]), {})
            topic_raw = clean_text(mapped.get("topic"))
            explanation_raw = clean_text(mapped.get("explanation"))

            if (
                topic_raw == OVERALL_TOPIC_LABEL
                and has_specific_reason(row["memo"], CLUE_KEYWORDS)
                and not should_be_overall(row["memo"], CLUE_KEYWORDS, GENERIC_TERMS)
            ):
                retry_result = call_llm(build_reclassify_without_overall_message(row))
                retry_topic = clean_text(retry_result.get("topic"))
                retry_explanation = clean_text(retry_result.get("explanation"))
                if retry_topic:
                    topic_raw = retry_topic
                if retry_explanation:
                    explanation_raw = retry_explanation

            batch_saved_rows.append(
                {
                    "_row_id": row["_row_id"],
                    "row_key": row["row_key"],
                    "memo": row["memo"],
                    "topic_raw": topic_raw,
                    "explanation_raw": explanation_raw,
                    "similarity_score": None,
                    "classification_route": "llm_batch",
                }
            )

    except Exception as batch_exc:
        print(f"[BATCH FALLBACK] batch_no={batch_no}, error={repr(batch_exc)}")

        for row in batch:
            try:
                row_result = classify_one_row_safely(row)
                batch_saved_rows.append(row_result)
            except Exception as row_exc:
                batch_failed_rows.append(
                    {
                        "_row_id": row["_row_id"],
                        "row_key": row["row_key"],
                        "memo": row["memo"],
                        "error_stage": "row_retry_failed",
                        "error_message": clean_text(str(row_exc)),
                    }
                )

    if batch_saved_rows:
        append_table(spark.createDataFrame(pd.DataFrame(batch_saved_rows)), INTERMEDIATE_REMAINING_TABLE)
        saved_total += len(batch_saved_rows)

    if batch_failed_rows:
        append_table(spark.createDataFrame(pd.DataFrame(batch_failed_rows)), INTERMEDIATE_FAILED_TABLE)
        failed_total += len(batch_failed_rows)

    processed_total += batch_size
    progress_pct = round(processed_total / total_rows_to_process * 100, 2) if total_rows_to_process > 0 else 100.0
    elapsed_sec = round(time.time() - batch_start_time, 2)

    print(
        f"[PROGRESS] batch_no={batch_no}/{total_batches}, "
        f"processed_total={processed_total}/{total_rows_to_process} ({progress_pct}%), "
        f"saved_total={saved_total}, failed_total={failed_total}, "
_row_id        f"batch_elapsed_sec={elapsed_sec}"
    )

    time.sleep(RATE_LIMIT_SECONDS)

print(
    f"[DONE] processed_total={processed_total}, saved_total={saved_total}, failed_total={failed_total}"
)


In [0]:
INTERMEDIATE_REMAINING_TABLE

In [0]:
# COMMAND ----------
# 14) Load Checkpoint Result And Combine With Sample Result

remaining_base_df = (
    spark.table(INTERMEDIATE_REMAINING_TABLE)
    .select("_row_id", "memo", "topic_raw", "explanation_raw")
    .dropDuplicates(["_row_id"])
)

remaining_classified_df = (
    remaining_base_df
    .withColumn("similarity_score", F.lit(None).cast("double"))
    .withColumn("classification_route", F.lit("llm_batch"))
    .select("_row_id", "memo", "topic_raw", "explanation_raw", "similarity_score", "classification_route")
)

sample_classified_df = (
    classified_df
    .withColumn("similarity_score", F.lit(None).cast("double"))
    .withColumn("classification_route", F.lit("sample_llm"))
    .select("_row_id", "memo", "topic_raw", "explanation_raw", "similarity_score", "classification_route")
)
combined_classified_df = sample_classified_df.unionByName(
    remaining_classified_df,
    allowMissingColumns=True
)

print(f"[SAMPLE CLASSIFIED] {sample_classified_df.count()}")
print(f"[REMAINING CLASSIFIED] {remaining_classified_df.count()}")
print(f"[COMBINED CLASSIFIED] {combined_classified_df.count()}")

display(combined_classified_df.limit(20))
combined_classified_df


In [0]:
# COMMAND ----------
# 15) Final Merge And Save

combined_topic_stats_df = (
    combined_classified_df.groupBy("topic_raw")
    .agg(F.count("*").alias("topic_cnt"))
)

combined_rare_topics = [
    r["topic_raw"]
    for r in combined_topic_stats_df.where(F.col("topic_cnt") < F.lit(MIN_ROWS_PER_TOPIC)).collect()
]
combined_rare_topics_sorted = sorted([t for t in combined_rare_topics if clean_text(t)])
combined_rare_topic_label = f"기타({', '.join(combined_rare_topics_sorted[:3])})" if combined_rare_topics_sorted else "기타"

topic_desc_map = {
    r["topic"]: r["description"]
    for r in topic_pool_df.select("topic", "description").toLocalIterator()
}

combined_final_rows = []
for row in combined_classified_df.toLocalIterator():
    row_dict = row.asDict(recursive=True)
    raw_topic = clean_text(row_dict["topic_raw"])
    raw_expl = clean_text(row_dict["explanation_raw"])

    if raw_topic in combined_rare_topics_sorted:
        final_topic = combined_rare_topic_label
        final_explanation = f"희소 주제 묶음: {raw_topic}" if raw_topic else "희소 주제 묶음"
    else:
        final_topic = raw_topic if raw_topic else "기타"
        final_explanation = raw_expl or topic_desc_map.get(final_topic, "")

    combined_final_rows.append(
        {
            "_row_id": row_dict["_row_id"],
            "memo": row_dict["memo"],
            "topic": final_topic,
            "description": final_explanation,
            "classification_route": row_dict["classification_route"],
            "similarity_score": row_dict["similarity_score"],
        }
    )

combined_detail_df = spark.createDataFrame(pd.DataFrame(combined_final_rows))

combined_summary_df = (
    combined_detail_df.groupBy("topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
    .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy(F.desc("review_count"), "topic")
)

save_table(combined_detail_df, FULL_DETAIL_TABLE)
save_table(combined_summary_df, FULL_SUMMARY_TABLE)

print("[FULL SUMMARY]")
display(combined_summary_df)

print("[FULL DETAIL]")
display(combined_detail_df)


In [0]:
# COMMAND ----------
# 16) Reclassify Over-Generalized Overall Positive

RECLASSIFIED_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_full_reoverall_{PROMPT_VERSION}"
RECLASSIFIED_SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_full_reoverall_{PROMPT_VERSION}"

STRICT_GENERIC_POSITIVE_TERMS = [
    "good", "great", "nice", "best", "excellent", "love it", "awesome",
    "좋아요", "좋다", "좋음", "최고", "만족", "괜찮아요", "굿", "나이스"
]

def is_strict_overall_positive(text: str) -> bool:
    memo = clean_text(text).lower()
    if not memo:
        return False

    # 문자/단어가 너무 길면 전반적 긍정 금지
    if len(memo) > 12:
        return False
    if len(re.findall(r"[A-Za-z가-힣]+", memo)) > 2:
        return False

    # generic praise만 허용
    normalized = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", memo)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    return any(term in normalized for term in STRICT_GENERIC_POSITIVE_TERMS)


non_overall_topic_payload = [
    {
        "topic": r["topic"],
        "description": r["description"],
    }
    for r in topic_pool_df.orderBy("topic_order").toLocalIterator()
    if clean_text(r["topic"]) != OVERALL_TOPIC_LABEL
]

def build_reclassify_overall_batch_messages(batch_rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    system = f"""
You are a VOC classifier for positive TV remote-usability reviews.

These memos were previously over-classified as '{OVERALL_TOPIC_LABEL}'.
Reclassify each memo into exactly one of:
- an existing non-overall topic
- or '기타' if none of the existing topics fit well

Important rules:
- Use '{OVERALL_TOPIC_LABEL}' only for ultra-short generic praise such as '좋아요', 'good', 'nice'.
- If the memo contains any usable clue, feature, reason, benefit, quantity, design, interface, AI, usability,
  convenience, function, technology, or contextual positivity, it must NOT remain '{OVERALL_TOPIC_LABEL}'.
- Examples that must NOT stay as '{OVERALL_TOPIC_LABEL}':
  - '2개 리모컨'
  - 'high tech remote'
  - 'remote is the best interface'
  - 'AI 리모컨까지 포함되어 아주 편리하네요.'
- Choose the closest existing topic if possible.
- If no existing topic is suitable, choose '기타'.
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Allowed topics:\n"
        + json.dumps(non_overall_topic_payload + [{"topic": "기타", "description": "기존 토픽에 명확히 맞지 않는 기타 긍정 사유"}], ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


overall_positive_rows = [
    r.asDict(recursive=True)
    for r in combined_detail_df.where(F.col("topic") == F.lit(OVERALL_TOPIC_LABEL)).toLocalIterator()
]

strict_keep_rows = []
to_reclassify_rows = []

for row in overall_positive_rows:
    if is_strict_overall_positive(row["memo"]):
        strict_keep_rows.append(
            {
                "_row_id": row["_row_id"],
                "memo": row["memo"],
                "topic": OVERALL_TOPIC_LABEL,
                "description": row["description"] if clean_text(row["description"]) else "초단문 일반 긍정 표현",
                "classification_route": row["classification_route"],
                "similarity_score": row["similarity_score"],
            }
        )
    else:
        to_reclassify_rows.append(row)

print(f"[OVERALL POSITIVE] total={len(overall_positive_rows)}")
print(f"[STRICT KEEP] {len(strict_keep_rows)}")
print(f"[TO RECLASSIFY] {len(to_reclassify_rows)}")

reclassified_rows = []

RECLASSIFY_BATCH_SIZE = 20

for batch_no, batch in enumerate(chunk_list(to_reclassify_rows, RECLASSIFY_BATCH_SIZE), start=1):
    print(f"[RECLASSIFY BATCH] {batch_no}, rows={len(batch)}")
    batch_result = call_llm(build_reclassify_overall_batch_messages(batch))
    result_map = {
        str(item.get("row_id")): item
        for item in batch_result.get("results", [])
        if isinstance(item, dict)
    }

    for row in batch:
        mapped = result_map.get(str(row["_row_id"]), {})
        new_topic = clean_text(mapped.get("topic"))
        new_expl = clean_text(mapped.get("explanation"))

        if not new_topic:
            new_topic = "기타"

        if new_topic == OVERALL_TOPIC_LABEL:
            new_topic = "기타"

        reclassified_rows.append(
            {
                "_row_id": row["_row_id"],
                "memo": row["memo"],
                "topic": new_topic,
                "description": new_expl or "전반적 긍정 재검토 후 재분류",
                "classification_route": "overall_recheck",
                "similarity_score": row["similarity_score"],
            }
        )

    time.sleep(RATE_LIMIT_SECONDS)


non_overall_existing_df = combined_detail_df.where(F.col("topic") != F.lit(OVERALL_TOPIC_LABEL))

strict_keep_df = spark.createDataFrame(pd.DataFrame(strict_keep_rows)) if strict_keep_rows else spark.createDataFrame(
    [], " _row_id long, memo string, topic string, description string, classification_route string, similarity_score double "
)

reclassified_df = spark.createDataFrame(pd.DataFrame(reclassified_rows)) if reclassified_rows else spark.createDataFrame(
    [], " _row_id long, memo string, topic string, description string, classification_route string, similarity_score double ")



In [0]:
# ─── Runtime Recovery: 테이블 → DataFrame 복원 ─────────────────────────────
# 셀 1, 2, 3 재실행 후 이 셀을 실행하세요

# 1) 테이블명 정의 (원래 셀 11에서 정의)
FULL_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_full_{PROMPT_VERSION}"
FULL_SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_full_{PROMPT_VERSION}"

# 2) TOPIC_POOL_TABLE → topic_pool_df
topic_pool_df = spark.table(TOPIC_POOL_TABLE)

# 3) RULE_PROFILE_TABLE → OVERALL_TOPIC_LABEL 등 규칙 변수
rule_profile_row = (
    spark.table(RULE_PROFILE_TABLE)
    .where(F.col("sc_measurement") == F.lit(TARGET_SC_MEASUREMENT))
    .limit(1)
    .collect()[0]
)
OVERALL_TOPIC_LABEL = rule_profile_row["overall_topic_label"]
CLUE_KEYWORDS = json.loads(rule_profile_row["clue_keywords_json"]) if rule_profile_row["clue_keywords_json"] else []
GENERIC_TERMS = json.loads(rule_profile_row["generic_terms_json"]) if rule_profile_row["generic_terms_json"] else []

# 4) FULL_DETAIL_TABLE → combined_detail_df
combined_detail_df = spark.table(FULL_DETAIL_TABLE)

print(f"[RECOVERY] topic_pool_df: {topic_pool_df.count()} topics")
print(f"[RECOVERY] OVERALL_TOPIC_LABEL = {OVERALL_TOPIC_LABEL}")
print(f"[RECOVERY] CLUE_KEYWORDS count = {len(CLUE_KEYWORDS)}")
print(f"[RECOVERY] combined_detail_df: {combined_detail_df.count()} rows")

In [0]:
phase1_detail_df = non_overall_existing_df.unionByName(strict_keep_df).unionByName(reclassified_df)

print(f"\n[PHASE 1 DONE] total={phase1_detail_df.count()}")

# ─── Phase 2: Reclassify 기타 ────────────────────────────────────────────

REMOTE_TERMS = [
    "remote", "remote control", "remotecontrol", "controller",
    "리모컨", "리모콘", "리모트", "컨트롤러", "조작기", "조종기",
    "magic remote", "one remote", "smart remote",
    "매직리모컨", "매직 리모컨", "원리모컨", "스마트리모컨",
]

REMOTE_POSITIVE_TERMS = [
    "good", "great", "nice", "best", "excellent", "love", "awesome",
    "amazing", "fantastic", "wonderful", "perfect", "superb", "cool",
    "좋아요", "좋다", "좋음", "좋아", "좋습니다", "좋네요",
    "최고", "만족", "괜찮아요", "괜찮다", "굿", "나이스", "훌륭", "대박",
]

FILLER_WORDS = {
    "is", "are", "was", "were", "the", "a", "an", "very", "so", "really",
    "quite", "pretty", "just", "also", "too", "it", "its", "this", "that",
    "my", "our", "i", "we", "be", "been", "has", "have", "had", "do",
    "정말", "진짜", "매우", "아주", "너무", "참", "되게", "엄청", "완전",
}


def is_remote_generic_positive(text: str) -> bool:
    """리모컨 지칭 + 긍정 감성 + filler만 있으면 True (구체적 이유 없음)"""
    memo = clean_text(text).lower()
    if not memo:
        return False

    has_remote = any(t in memo for t in REMOTE_TERMS)
    if not has_remote:
        return False

    has_positive = any(t in memo for t in REMOTE_POSITIVE_TERMS)
    if not has_positive:
        return False

    remaining = memo
    for term in sorted(REMOTE_TERMS, key=len, reverse=True):
        remaining = remaining.replace(term, " ")
    for term in sorted(REMOTE_POSITIVE_TERMS, key=len, reverse=True):
        remaining = remaining.replace(term, " ")

    words = re.findall(r"[a-zA-Z가-힣]+", remaining)
    meaningful = [w for w in words if w not in FILLER_WORDS and len(w) > 1]

    return len(meaningful) == 0


phase2_topic_payload = [
    {"topic": r["topic"], "description": r["description"]}
    for r in topic_pool_df.orderBy("topic_order").toLocalIterator()
] + [
    {"topic": "오분류", "description": "리모컨이 아닌 다른 카테고리(화질, 음질, 화면, 배송 등)에 대한 칭찬이거나, 부정적/불만 문장"},
]


def build_reclassify_etc_batch_messages(batch_rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    system = f"""You are a VOC classifier for TV remote-usability reviews.

These memos were classified as '기타' (Others). Reclassify each into exactly one category.

Rules:
1. '전반적 긍정': The memo ONLY says the remote is good with NO specific reason/feature/benefit.
   Examples: "remote is great", "리모컨 좋아요", "love the remote"

2. '오분류': The memo praises something OTHER than the remote (picture, sound, screen, delivery, etc.)
   OR the memo is negative/complaining about the remote.
   Examples: "화질이 정말 좋아요", "remote doesn't work", "great sound quality"

3. Specific topic: The memo hints at WHY the remote is good. Classify into:
   - An existing topic from the list if it fits
   - OR a new concise Korean label if no existing topic fits
     (e.g., "기능만족", "이용성만족", "수량만족", "기술만족", "호환성만족", "가성비만족")
   Even minimal clues count.
   Examples:
     "it has many built in smart features" → 앱 바로가기·전용 버튼 or 기능만족
     "remote control is very useful" → 직관적 조작·쉬운 사용
     "2 remote controls are useful" → 수량만족
     "The remote feels premium in hand" → 그립감·크기·무게·디자인
     "high tech remote" → 기술만족

Important:
- Prefer existing topics when a reasonable match exists.
- New labels must be concise Korean (2-5 chars + optional qualifier).
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Allowed topics:\n"
        + json.dumps(phase2_topic_payload, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


etc_condition = (F.col("topic") == F.lit("기타")) | F.col("topic").like("기타(%")

phase1_etc_rows = [
    r.asDict(recursive=True)
    for r in phase1_detail_df.where(etc_condition).toLocalIterator()
]

phase2_generic_positive_rows = []
phase2_to_reclassify_rows = []

for row in phase1_etc_rows:
    if is_remote_generic_positive(row["memo"]):
        phase2_generic_positive_rows.append({
            "_row_id": row["_row_id"],
            "memo": row["memo"],
            "topic": OVERALL_TOPIC_LABEL,
            "description": "리모컨에 대한 구체적 이유 없는 일반 긍정",
            "classification_route": "etc_rule_generic_positive",
            "similarity_score": row["similarity_score"],
        })
    else:
        phase2_to_reclassify_rows.append(row)

print(f"\n[PHASE 2] 기타 total={len(phase1_etc_rows)}")
print(f"[PHASE 2] rule → 전반적 긍정: {len(phase2_generic_positive_rows)}")
print(f"[PHASE 2] to reclassify by LLM: {len(phase2_to_reclassify_rows)}")

PHASE2_BATCH_SIZE = 20
phase2_reclassified_rows = []

for batch_no, batch in enumerate(chunk_list(phase2_to_reclassify_rows, PHASE2_BATCH_SIZE), start=1):
    print(f"[PHASE 2 BATCH] {batch_no}, rows={len(batch)}")
    batch_result = call_llm(build_reclassify_etc_batch_messages(batch))
    result_map = {
        str(item.get("row_id")): item
        for item in batch_result.get("results", [])
        if isinstance(item, dict)
    }

    for row in batch:
        mapped = result_map.get(str(row["_row_id"]), {})
        new_topic = clean_text(mapped.get("topic"))
        new_expl = clean_text(mapped.get("explanation"))

        if not new_topic:
            new_topic = "기타"

        phase2_reclassified_rows.append({
            "_row_id": row["_row_id"],
            "memo": row["memo"],
            "topic": new_topic,
            "description": new_expl or "기타 재분류",
            "classification_route": "etc_recheck",
            "similarity_score": row["similarity_score"],
        })

    time.sleep(RATE_LIMIT_SECONDS)


# ─── Final merge ─────────────────────────────────────────────────────────

non_etc_df = phase1_detail_df.where(~etc_condition)

phase2_generic_df = (
    spark.createDataFrame(pd.DataFrame(phase2_generic_positive_rows))
    if phase2_generic_positive_rows
    else spark.createDataFrame(
        [], "_row_id long, memo string, topic string, description string, classification_route string, similarity_score double"
    )
)

phase2_reclass_df = (
    spark.createDataFrame(pd.DataFrame(phase2_reclassified_rows))
    if phase2_reclassified_rows
    else spark.createDataFrame(
        [], "_row_id long, memo string, topic string, description string, classification_route string, similarity_score double"
    )
)

reoverall_detail_df = non_etc_df.unionByName(phase2_generic_df).unionByName(phase2_reclass_df)


# ─── Summary & Save ──────────────────────────────────────────────────────

reoverall_summary_df = (
    reoverall_detail_df.groupBy("topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
    .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy(F.desc("review_count"), "topic")
)

save_table(reoverall_detail_df, RECLASSIFIED_DETAIL_TABLE)
save_table(reoverall_summary_df, RECLASSIFIED_SUMMARY_TABLE)

print("\n[FINAL SUMMARY]")
display(reoverall_summary_df)

print("[FINAL DETAIL]")
display(reoverall_detail_df)